# Notebook 32 — Real-model temperature and sampling-budget sensitivity

This is the missing **real model** control. Notebook 30 varies `K` and response bias `p` only for a signal-free mathematical null; it does not show how the transcription model behaves under other generation settings. Here Qwen2.5-VL-3B is run on the exact frozen FERMAT `n=300` pages at temperatures `{0.3, 0.7, 1.0}`. Each condition draws ten responses, and nested prefixes give `K in {3,5,10}` without a separate run for every K.

## Fixed interpretation contract

- The declared grid is a sensitivity analysis, not hyperparameter tuning. No condition is selected as a new headline winner.
- The experiment is perception/transcription only. It does not revive the invalid stratified grading result.
- Every condition uses the same model, prompt, pages, scorer, and item order.
- The notebook first performs the authenticated Stage-B reconstruction against pinned FERMAT revision `80ff9934c38615bb8d3a33c24252db02e21774f0`. It fails before model loading unless all 300 rows match the frozen run **in order**.
- Item-level text, images, and raw generations remain private on Drive. Only aggregate metrics, configuration, hashes, and a figure may enter the public artifact.
- `PROCESS_N=300` is required for a paper result. Any smaller value is a smoke run and is saved under a different name.

## Inputs and outputs

Input: the private frozen run CSV on Drive and authorized FERMAT access. Output: resumable private JSONL checkpoints on Drive; a public aggregate CSV/JSON/PNG under `reference/wacv_evaluation_artifact/` in the clone and on Drive. No second rater is used or implied. GPU cost is 9,000 transcription generations for the full grid; checkpoints are written after every item.


In [1]:
# Colab environment, private token store, and commit-specific checkout.
%pip install -q transformers accelerate datasets huggingface_hub qwen-vl-utils sympy pytest antlr4-python3-runtime==4.11

import hashlib
import importlib
import json
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

from google.colab import drive
from huggingface_hub import login

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/uncertainty-math-vlm')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
TOKEN_FILE = PROJECT_DIR / '.tokens.json'
RESET_TOKENS = False

def get_token(name, prompt):
    tokens = json.loads(TOKEN_FILE.read_text()) if TOKEN_FILE.exists() else {}
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        TOKEN_FILE.write_text(json.dumps(tokens))
        os.chmod(TOKEN_FILE, 0o600)
    return tokens[name]

HF_TOKEN = get_token('HF_TOKEN', 'Hugging Face token (asked once): ')
if not HF_TOKEN.startswith('hf_'):
    raise ValueError('Stored HF token is invalid; set RESET_TOKENS=True and replace it.')
login(token=HF_TOKEN)

REPO_URL = 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git'
remote = subprocess.run(['git', 'ls-remote', REPO_URL, 'refs/heads/main'],
                        check=True, capture_output=True, text=True).stdout.split()[0]
REPO_DIR = Path(f'/content/nb32_repo_{remote[:7]}')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
head = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
                      check=True, capture_output=True, text=True).stdout.strip()
assert head == remote, f'stale checkout: {head} != {remote}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()
for name in [m for m in list(sys.modules) if m == 'pilot' or m.startswith('pilot.')]:
    del sys.modules[name]

import pilot.parameter_sensitivity as PS
import pilot.prompts as prompts
import pilot.wacv_artifact as W
print('running commit:', head)
print('pilot code:', Path(PS.__file__).parent)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
running commit: 87a0fb96f8f4504bec4bb0a61bc25ff524b63a3c
pilot code: /content/nb32_repo_87a0fb9/pilot


In [2]:
# Authenticated Stage B: reconstruct the exact sample and update only safe public files.
import datasets
import pandas as pd

PINNED_REVISION = '80ff9934c38615bb8d3a33c24252db02e21774f0'
RUN_NAME = 'scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv'
RUN_CSV = PROJECT_DIR / 'results' / RUN_NAME
EXPECTED_RUN_SHA256 = 'f8ffc23fbc2c038f98d0fa82fc96edfbe516fce5466de90d741a2a8e3e9a1336'
assert RUN_CSV.exists(), f'private frozen run missing: {RUN_CSV}'
assert W.sha256_file(str(RUN_CSV)) == EXPECTED_RUN_SHA256, 'frozen run hash changed'
run = pd.read_csv(RUN_CSV)
assert len(run) == 300

fermat = datasets.load_dataset(W.DATASET_ID, split=W.DATASET_SPLIT,
                               revision=PINNED_REVISION, token=HF_TOKEN)
source = W.reconstruct_balanced_source(fermat, n=300, seed=42, target_error_frac=0.5)
alignment = W.verify_reconstruction(run, source=source, revision=PINNED_REVISION)
print(alignment['alignment_status'], alignment['n_aligned'], '/', alignment['n_expected'])
assert alignment['verified'], alignment['reason']

# Materialize images in the verified final order. source_row_index refers to
# the pinned dataset before either shuffle.
sample = fermat.select(source['source_row_index'].astype(int).tolist())
assert all(str(sample[i]['orig_q']) == str(run.loc[i, 'orig_q']) for i in range(300))
assert all(str(sample[i]['pert_a']) == str(run.loc[i, 'pert_a']) for i in range(300))

revision = {'dataset_id': W.DATASET_ID, 'revision': PINNED_REVISION,
            'gated': 'auto', 'source': 'authenticated pinned load_dataset'}
audit_long = W.audit_labels_long(str(REPO_DIR / 'reference/audit'))
public, private = W.build_manifests(run, str(RUN_CSV), audit_long, revision, alignment)
W.assert_public_manifest_is_safe(public, private)

PUBLIC_REPO = REPO_DIR / 'reference/wacv_evaluation_artifact'
PUBLIC_DRIVE = PROJECT_DIR / 'wacv_public_artifacts'
PRIVATE_DRIVE = PROJECT_DIR / 'wacv_private_artifacts'
for path in (PUBLIC_REPO, PUBLIC_DRIVE, PRIVATE_DRIVE):
    path.mkdir(parents=True, exist_ok=True)
public.to_csv(PUBLIC_REPO / 'fermat_n300_public_manifest.csv', index=False)
public.to_csv(PUBLIC_DRIVE / 'fermat_n300_public_manifest.csv', index=False)
private.to_csv(PRIVATE_DRIVE / 'fermat_n300_private_manifest.csv', index=False)

prov_path = PUBLIC_REPO / 'provenance.json'
provenance = json.loads(prov_path.read_text())
provenance['alignment'] = {k: v for k, v in alignment.items() if k != 'source_row_index'}
provenance['dataset'] = revision
provenance['protocol'] = W.RUN_PROTOCOL
prov_path.write_text(json.dumps(provenance, indent=2, sort_keys=True))
(PUBLIC_DRIVE / 'provenance.json').write_text(prov_path.read_text())
assert provenance['protocol']['temperature'] == 0.7
print('authenticated reconstruction VERIFIED; public manifest/provenance updated')


VERIFIED 300 / 300
authenticated reconstruction VERIFIED; public manifest/provenance updated


In [3]:
# Run validation on Colab, not on the local laptop. These are CPU tests; the
# GPU is used below for inference. The focused gate covers every file changed
# for Notebook 32: 49 tests, measured at 12 s.
#
# RUN_FULL_CPU_TESTS runs all 760 tests instead. Measured at 53 MINUTES, and it
# adds no coverage of anything this notebook changed. It also spends that time
# inside a metered GPU session while the GPU sits idle. Off by default; turn it
# on only to revalidate the whole repo with no GPU work queued behind it.
RUN_FULL_CPU_TESTS = False

import time

repo_results = REPO_DIR / 'results'
repo_results.mkdir(exist_ok=True)
test_run = repo_results / RUN_NAME
if not test_run.exists():
    test_run.symlink_to(RUN_CSV)


def run_step(label, argv, expect):
    """Announce, time and stream each step; `check=True` still aborts on failure."""
    print(f'\n=== {label}  (expect ~{expect}) ===', flush=True)
    started = time.time()
    subprocess.run(argv, cwd=REPO_DIR, check=True)
    print(f'--- {label} finished in {time.time() - started:.0f} s', flush=True)


# python -u: without it pytest's progress arrives in one block at the very end,
# so a passing run is indistinguishable from a hung one while it is going.
run_step('focused tests', [
    sys.executable, '-u', '-m', 'pytest', '-q',
    'pilot/tests/test_wacv_evaluation_artifact.py',
    'pilot/tests/test_parameter_sensitivity.py',
    'pilot/tests/test_confidence_and_variants.py',
], '12 s')

run_step('notebook 32 dry run',
         [sys.executable, '-u', 'pilot/dryruns/dryrun_nb32.py'], '5 s')

if RUN_FULL_CPU_TESTS:
    run_step('full suite, 760 tests',
             [sys.executable, '-u', '-m', 'pytest', 'pilot/tests', '-q'], '53 min')

print('\nNotebook 32 validation PASS')



=== focused tests  (expect ~12 s) ===
--- focused tests finished in 5 s

=== notebook 32 dry run  (expect ~5 s) ===
--- notebook 32 dry run finished in 2 s

Notebook 32 validation PASS


In [4]:
# Model and declared grid. PROCESS_N=300 is the paper run; lower is smoke only.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
TEMPERATURES = PS.TEMPERATURES
K_VALUES = PS.K_VALUES
K_MAX = PS.K_MAX
PROCESS_N = 300
MAX_NEW_TOKENS = 512
SAMPLE_BATCH = 5
SCORER = 'strict_v1'

MODEL_CACHE = PROJECT_DIR / 'model_cache'
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=MODEL_CACHE)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', cache_dir=MODEL_CACHE)
model.eval()
print('GPU:', torch.cuda.get_device_name(0))
print('grid:', TEMPERATURES, 'x', K_VALUES, 'items:', PROCESS_N)
print('new generations:', len(TEMPERATURES) * K_MAX * PROCESS_N)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

GPU: NVIDIA A100-SXM4-40GB
grid: (0.3, 0.7, 1.0) x (3, 5, 10) items: 300
new generations: 9000


In [5]:
# Deterministic, resumable generation. Raw outputs stay in private Drive checkpoints.
import gc
import time
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints' / 'nb32_parameter_sensitivity'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def qwen_inputs(messages):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    return processor(text=[text], images=images, videos=videos, padding=True,
                     return_tensors='pt').to(model.device)

def generate_exact_k(messages, temperature, seed):
    # Seed is reset once per item/temperature, so resume does not change draws.
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    inputs = qwen_inputs(messages)
    texts = []
    try:
        while len(texts) < K_MAX:
            n = min(SAMPLE_BATCH, K_MAX - len(texts))
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                     do_sample=True, temperature=temperature,
                                     num_return_sequences=n)
            trimmed = out[:, inputs['input_ids'].shape[1]:]
            texts.extend(processor.batch_decode(trimmed, skip_special_tokens=True,
                                                clean_up_tokenization_spaces=False))
            del out, trimmed
        return texts
    finally:
        del inputs
        gc.collect()
        torch.cuda.empty_cache()

def checkpoint_paths(temperature):
    slug = str(temperature).replace('.', 'p')
    return (CHECKPOINT_DIR / f'temp_{slug}_k{K_MAX}_n300.jsonl',
            CHECKPOINT_DIR / f'temp_{slug}_k{K_MAX}_n300_config.json')

# Settings that CHANGE THE SAMPLES are compared strictly on resume.
# repo_commit is recorded but NOT compared: an unrelated commit (a docs fix,
# another notebook) would otherwise reject a multi-hour checkpoint and force a
# full regeneration. What actually drives generation is pinned instead by
# hashing pilot/parameter_sensitivity.py, which is where cell_seed lives.
PROVENANCE_ONLY = ('repo_commit',)


def expected_config(temperature):
    return {'model_id': MODEL_ID, 'temperature': temperature, 'k_max': K_MAX,
            'sample_n': 300, 'selection_seed': 42,
            'generation_seed': PS.BASE_SEED, 'sample_batch': SAMPLE_BATCH,
            'max_new_tokens': MAX_NEW_TOKENS, 'prompt_sha256':
            W.sha256_text(prompts.TRANSCRIPTION_USER_PROMPT),
            'dataset_revision': PINNED_REVISION,
            'sampler_code_sha256': W.sha256_file(PS.__file__)}

def load_checkpoint(temperature):
    data_path, config_path = checkpoint_paths(temperature)
    config = expected_config(temperature)
    if config_path.exists():
        stored = json.loads(config_path.read_text())
        strict = {k: v for k, v in stored.items() if k not in PROVENANCE_ONLY}
        # A key ABSENT from a stored config means it was written before that key
        # existed, not that the setting disagrees. Discarding an otherwise
        # identical run over a schema addition would throw away hours of GPU for
        # nothing. Such keys are skipped here and the file is upgraded below;
        # every entry's seed is re-derived further down, which verifies the
        # sampler directly and is stronger than the hash being added.
        added = [k for k in config if k not in strict]
        differs = sorted(k for k in set(strict) | set(config)
                         if k not in added and strict.get(k) != config.get(k))
        assert not differs, (
            f'checkpoint config mismatch at {config_path}; do not mix runs. '
            f'differs on: {differs}')
        if added:
            config_path.write_text(json.dumps({**config, 'repo_commit': head},
                                              indent=2, sort_keys=True))
            print(f'checkpoint config upgraded, added {added}; '
                  'all pre-existing settings matched')
        if stored.get('repo_commit') != head:
            print(f"NOTE: checkpoint was written at repo_commit "
                  f"{stored.get('repo_commit')}, now {head}. Every sampling "
                  'setting matches, so the draws stay comparable and this '
                  'resume is safe.')
    else:
        config_path.write_text(json.dumps({**config, 'repo_commit': head},
                                          indent=2, sort_keys=True))
    entries = []
    if data_path.exists():
        for line in data_path.read_text().splitlines():
            if not line.strip():
                continue
            entry = json.loads(line)
            i = len(entries)
            expected_hash = W.sha256_text(f"{run.loc[i, 'orig_q']}\x1f{run.loc[i, 'pert_a']}")
            assert entry['item_id'] == i and entry['content_sha256'] == expected_hash
            assert len(entry['raw_samples']) == K_MAX
            # Re-derive the seed rather than trusting the recorded one. This is
            # the direct test that the sampler is unchanged, it works on
            # checkpoints written before any code hash was stored, and it is
            # what sampler_code_sha256 is only a proxy for.
            assert entry['seed'] == PS.cell_seed(temperature, i), (
                f"entry {i} seed {entry['seed']} != "
                f'{PS.cell_seed(temperature, i)}; the sampler changed')
            entries.append(entry)
    assert len(entries) <= PROCESS_N
    return data_path, entries


In [6]:
# Full grid. Safe to rerun: every completed item resumes from Drive.
raw_by_temperature = {}
for temperature in tqdm(TEMPERATURES, desc='grid', unit='temp'):
    data_path, entries = load_checkpoint(temperature)
    print(f'\ntemperature={temperature}: resuming at {len(entries)}/{PROCESS_N}')
    with tqdm(total=PROCESS_N, initial=len(entries), desc=f'temp={temperature}', unit='item') as bar:
        for item_id in range(len(entries), PROCESS_N):
            item = sample[item_id]
            started = time.time()
            seed = PS.cell_seed(temperature, item_id)
            raw = generate_exact_k(prompts.build_transcription_messages(item['image']),
                                   temperature, seed)
            entry = {
                'item_id': item_id, 'temperature': temperature, 'seed': seed,
                'content_sha256': W.sha256_text(
                    f"{run.loc[item_id, 'orig_q']}\x1f{run.loc[item_id, 'pert_a']}"),
                'raw_samples': raw, 'elapsed_seconds': time.time() - started,
            }
            with data_path.open('a') as fh:
                fh.write(json.dumps(entry) + '\n')
                fh.flush()
            entries.append(entry)
            bar.update(1)
    raw_by_temperature[temperature] = entries

assert all(len(v) == PROCESS_N for v in raw_by_temperature.values())
print('generation grid complete')


grid:   0%|          | 0/3 [00:00<?, ?temp/s]

AssertionError: checkpoint config mismatch at /content/drive/MyDrive/uncertainty-math-vlm/checkpoints/nb32_parameter_sensitivity/temp_0p3_k10_n300_config.json; do not mix runs. differs on: ['sampler_code_sha256']

In [ ]:
# Score every nested prefix and apply the preregistered completeness/power gate.
import pilot.canonicalize as canonicalize
from tqdm.auto import tqdm

# ~330 ms per item on the reference machine, so a full 3x300 grid is about
# five minutes. Unbarred that is five minutes of silence immediately after a
# multi-hour generation run, which is indistinguishable from a hang.
scored_rows = []
to_score = sum(len(v) for v in raw_by_temperature.values())
with tqdm(total=to_score, desc='scoring', unit='item') as bar:
    for temperature, entries in raw_by_temperature.items():
        for entry in entries:
            item_id = entry['item_id']
            scored_rows.extend(PS.score_nested_samples(
                entry['raw_samples'], run.loc[item_id, 'pert_a'], temperature, item_id,
                k_values=K_VALUES, rule=SCORER))
            bar.update(1)
scored = pd.DataFrame(scored_rows)
boot = 10_000 if PROCESS_N == 300 else 500
print(f'bootstrapping {len(TEMPERATURES) * len(K_VALUES)} grid cells '
      f'at n_boot={boot} (about 12 s at 300 items)')
summary = PS.summarize_grid(scored, temperatures=TEMPERATURES, k_values=K_VALUES,
                            n_items=PROCESS_N, n_boot=boot, seed=0)
gate = PS.reviewer_gate(summary)
display(summary.round(4))
print(gate)
print('LaTeX parser available:', canonicalize.latex_parser_available())
if PROCESS_N < 300:
    assert not gate['paper_eligible'], 'a smoke run must never become paper eligible'


In [ ]:
# Public aggregate artifacts; private item scores stay on Drive. No automatic push.
import matplotlib.pyplot as plt

status = 'full' if gate['paper_eligible'] else f'smoke_n{PROCESS_N}'
stem = f'real_parameter_sensitivity_{status}'
out_repo = PUBLIC_REPO
out_drive = PUBLIC_DRIVE
private_scored = PRIVATE_DRIVE / f'{stem}_item_scores.csv'
scored.to_csv(private_scored, index=False)

config = {
    'status': status, 'gate': gate, 'model_id': MODEL_ID,
    'dataset_revision': PINNED_REVISION, 'alignment_status': alignment['alignment_status'],
    'n_items': PROCESS_N, 'temperatures': list(TEMPERATURES),
    'k_values': list(K_VALUES), 'nested_prefixes': True,
    'temperature_0p7_is_frozen_protocol': True, 'scorer': SCORER,
    'generation_seed': PS.BASE_SEED, 'sample_batch': SAMPLE_BATCH,
    'max_new_tokens': MAX_NEW_TOKENS, 'repo_commit': head,
    'prompt_sha256': W.sha256_text(prompts.TRANSCRIPTION_USER_PROMPT),
    'latex_parser_available': canonicalize.latex_parser_available(),
    'interpretation': gate['interpretation'],
    'forbidden': ['selecting the best cell as a tuned headline',
                  'claiming robustness outside the declared grid',
                  'treating frozen-scorer correctness as human truth'],
}

fig, ax = plt.subplots(figsize=(6.2, 4.0))
for k, group in summary.groupby('k'):
    group = group.sort_values('temperature')
    yerr = [group.auroc - group.ci_low, group.ci_high - group.auroc]
    ax.errorbar(group.temperature, group.auroc, yerr=yerr, marker='o', capsize=3, label=f'K={k}')
ax.axhline(0.5, color='0.55', linestyle='--', linewidth=1)
ax.set(xlabel='sampling temperature', ylabel='AUROC against frozen scorer',
       title='Real-model parameter sensitivity (Qwen2.5-VL-3B)')
ax.set_xticks(TEMPERATURES)
ax.legend(frameon=False)
fig.tight_layout()

for directory in (out_drive, out_repo):
    summary.to_csv(directory / f'{stem}.csv', index=False)
    (directory / f'{stem}.json').write_text(json.dumps(config, indent=2, sort_keys=True))
    fig.savefig(directory / f'{stem}.png', dpi=220, bbox_inches='tight')
plt.show()

print('\nHANDOFF')
print('  reconstruction:', alignment['alignment_status'])
print('  paper gate:', gate)
print('  public Drive:', out_drive)
print('  private item scores:', private_scored)
print('  clone files to copy:', out_repo)
print('  No second-rater result is claimed.')
print('  Review the full aggregate results before adding any temperature claim to the paper.')
